## Dev Notebook for subsetting HYDRO30 data by HydroBASIN

https://www.hydrosheds.org/products/hydrobasins

TODOs:
- Include lakes for now. We may change our minds.
- Handle already present data when downloading HydroBASINs and writing shapefiles
- Organize and put in a script
    - water mask path
    - buffer size
    - optional arg `continent` to reduce overhead when intersecting AOI with HydroBASINs
    - optional? HydroBASIN level
- If possible, add option to dynamically estimate a per-watershed buffer


## Fetch all the HydroBASIN data

### Define and create a directory to hold HydroBASIN data

In [ ]:
from pathlib import Path
import requests

hydrobasins_path = Path.cwd() / "HydroBASINS"
hydrobasins_path.mkdir(exist_ok=True)

### Download the data

In [ ]:
for continent in ["af", "ar", "as", "au", "eu", "gr", "na", "sa", "si"]:
    file = f"hybas_{continent}_lev01-12_v1c.zip"

    url = f"https://data.hydrosheds.org/file/hydrobasins/standard/{file}"
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        with open(hydrobasins_path/file, "wb") as f:
            f.write(response.content)
        print(f"Download complete: {url}")
    else:
        print(f"Failed to download: {response.status_code} - {response.reason}")

    !unzip {hydrobasins_path/file} -d {hydrobasins_path/Path(file).stem}
    (hydrobasins_path/file).unlink()

### Glob the paths to the level-12 data

In [ ]:
basin_12_paths = list(hydrobasins_path.glob("hybas_*/hybas_*_lev12_*.shp"))
basin_12_paths

### Put level-12 geometries for all continents in a geodataframe

In [ ]:
%%time

import geopandas as gpd
import pandas as pd
from shapely.ops import unary_union

gdfs = []
for path in basin_12_paths:
    gdf = gpd.read_file(path)
    gdf["filepath"] = path
    gdfs.append(gdf)


gdf = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True))

# repair self-intersections in geometries
gdf["geometry"] = gdf["geometry"].apply(
    lambda g: g.buffer(0) if g and not g.is_valid else g
)
gdf

### Confirm validity of geometries

HydroBASINS geometries contain many self-intersections, which must be removed prior to intersecting geometries in the gdf.

Confirm that all geometries are now valid.

In [ ]:
from shapely.validation import explain_validity

all("Valid" in v for v in explain_validity(gdf.geometry))

### Define a path to an RTC

In [ ]:
tiff_pth = Path("/home/jovyan/Kas/TX_flood_25/Water_Mask/water_extent_20250308.tif")

### Intersect the RTC's geometry with the HydroBASIN geometries

In [ ]:
from shapely.geometry import box
import rasterio

with rasterio.open(tiff_pth) as src:
    raster_bounds = src.bounds
    raster_crs = src.crs

raster_geom = box(*raster_bounds)
raster_gdf = gpd.GeoDataFrame({"geometry": [raster_geom]}, crs=raster_crs)
raster_gdf = raster_gdf.to_crs(gdf.geometry.crs)


gdf = gpd.overlay(gdf, raster_gdf, how="intersection")
gdf

### Project HydroBASIN geometries to RTC's CRS prior to buffering

Buffering in WGS84 will use a distance unit of degrees, which vary depending on latitude. This can lead to distortions and unexpected results.

Projecting to UTM prior to buffering allows us to use meters distance units and will not cause distortions. 

In [ ]:
gdf = gdf.to_crs(raster_crs)
gdf

In [ ]:
gdf.geometry.iloc[0]

### Buffer HydroBASIN geometries

In [ ]:
# TODO dynamically determine an ideal buffer size for each geometry?

# cap_style: {‘round’, ‘square’, ‘flat’}, default ‘round’
# join_style: {‘round’, ‘mitre’, ‘bevel’}, default ‘round’

buffer_m = 500

gdf["geometry"] = gdf["geometry"].apply(lambda g: g.buffer(buffer_m) if g else g)

In [ ]:
gdf.geometry.iloc[0]

## Write the buffered geometries to a new shapefile

In [ ]:
output_shapefile = tiff_pth.parents[1] / f"HydroBASIN_buffered_{tiff_pth.stem}"
gdf.to_file(output_shapefile)

### Throw away any buffered geometries that do not fall within the valid raster data area of the scene

In [ ]:
import numpy as np
import xarray as xr
import rioxarray
import rasterio.features
from shapely.geometry import shape, Polygon
from shapely.ops import unary_union
import geopandas as gpd

raster = rioxarray.open_rasterio(tiff_pth) #, chunks=True)
data_mask = raster.isel(band=0).values != raster.rio.nodata

shapes = rasterio.features.shapes(
    data_mask.astype(np.uint8),
    transform=raster.rio.transform()
)
valid_shapes = [shape(geom) for geom, val in shapes if val == 1]

valid_data_envelope = max(valid_shapes, key=lambda geom: geom.area)
valid_data_envelope = valid_data_envelope.exterior
valid_data_envelope = Polygon(valid_data_envelope)

gdf = gdf[gdf.geometry.within(valid_data_envelope)]
gdf.to_file("buffered_hydroBASINs.shp")
gdf

### Plot the buffered watershed geometries 

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

clipped_shapes_gdf = gpd.read_file("buffered_hydroBASINs.shp")


ax = clipped_shapes_gdf.boundary.plot(figsize=(10, 10), edgecolor='blue', linewidth=0.5)
valid_mask_gdf = gpd.GeoDataFrame(geometry=[valid_data_envelope], crs=clipped_shapes_gdf.crs)
valid_mask_gdf.boundary.plot(ax=ax, edgecolor='black', linewidth=2)

plt.title("Footprints of Clipped Watersheds within Valid Data Area")
plt.show()


In [ ]:
## TODO Add rioxarry to hydrosar environment

# !mamba install -c conda-forge rioxarray --yes

### Build an xarray.Dataset of raster data trimmed to the extents of the buffered Level-12 HydroBASINs 

In [ ]:
%%time

import dask
from dask import delayed
import numpy as np
import xarray as xr
import rioxarray
import geopandas as gpd
from tqdm.auto import tqdm

from itertools import islice

raster = rioxarray.open_rasterio(tiff_pth, chunks=True).squeeze('band')
raster_transform = raster.rio.transform()

@delayed
def clip_one(polygon, idx):
    try:
        clipped = raster.rio.clip([polygon], all_touched=True, drop=True)
        clipped = clipped.rio.write_crs(raster_crs, inplace=False)
        clipped = clipped.rio.write_transform(raster_transform, inplace=False)
        print(f"{idx}: CRS={clipped.rio.crs}, transform={clipped.rio.transform()}")

        return (f"polygon_{idx}", clipped)
    except Exception as e:
        return (f"polygon_{idx}", None)

tasks = []

for row in tqdm(gdf.itertuples(), total=len(gdf)):
    tasks.append(clip_one(row.geometry, row.Index))

results = dask.compute(*tasks)

clipped_data = {region_id: data for region_id, data in results if data is not None}
dataset = xr.Dataset(clipped_data)
dataset

In [ ]:
dataset.polygon_9.min()

In [ ]:
dataset.polygon_7.max()